<a href="https://colab.research.google.com/github/Aniyarath/AI-APPS/blob/main/day1_demo_code_list.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinInsight AI – Full Data Pipeline Demo
Upload PDFs (statements) and receipt images, then clean, structure, chunk, and embed data.

In [ ]:
# Upload files
from google.colab import files
uploaded = files.upload()

Saving bank_statement_5.pdf to bank_statement_5.pdf
Saving bank_statement_4.pdf to bank_statement_4.pdf
Saving bank_statement_3.pdf to bank_statement_3.pdf
Saving bank_statement_2.pdf to bank_statement_2.pdf
Saving bank_statement_1.pdf to bank_statement_1.pdf
Saving receipt_image_3.png to receipt_image_3.png
Saving receipt_image_2.png to receipt_image_2.png
Saving receipt_image_1.png to receipt_image_1.png


## Install Dependencies

In [ ]:
!pip install pdfplumber pytesseract sentence-transformers pandas openpyxl
!apt-get install -y tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


## OCR: Extract Text from Receipt Images

In [ ]:
import pytesseract
from PIL import Image

image_texts = []

for filename in uploaded:
    if filename.endswith('.png') or filename.endswith('.jpg'):
        img = Image.open(filename)
        text = pytesseract.image_to_string(img)
        image_texts.append(text)

image_texts[:2]

['STORE RECEIPT\n\nDate: April 2026\n\nTaxi Fare $15\n\nHeadphones $120,\n\nSnacks $8\n\nHeadphones $120,\n\nColes $4\n\nColes $4\n\nTOTAL $68\n\x0c',
 'STORE RECEIPT\n\nDate: April 2026\n\nColes $4\n\nBread $3\n\nColes $4\n\nColes $4\n\nTOTAL $121\n\x0c']

## PDF Extraction

In [ ]:
import pdfplumber

pdf_texts = []

for filename in uploaded:
    if filename.endswith('.pdf'):
        with pdfplumber.open(filename) as pdf:
            text = ''
            for page in pdf.pages:
                text += page.extract_text() or ''
        pdf_texts.append(text)

pdf_texts[:1]

['FinBank Credit Card Statement\nAccount Holder: John Doe\nStatement Period: April 1 - April 30, 2026\nDate Merchant Category Amount\nApr 21 Best Buy Shopping $142.29\nApr 13 Apple Store Shopping $100.31\nApr 20 Apple Store Shopping $101.5\nApr 20 Uber Transport $291.3\nApr 14 Uber Transport $129.09\nApr 11 Shell Gas $57.86\nApr 23 Target Shopping $55.86\nApr 13 Costco Groceries $240.85\nApr 10 Walmart Shopping $67.58\nApr 19 Costco Groceries $283.95\nApr 12 Costco Groceries $39.57\nApr 22 Lyft Transport $211.88\nApr 16 Netflix Subscription $290.12\nThank you for banking with FinBank.']

## Clean and Structure Data

In [ ]:
import re
import pandas as pd

all_lines = []

for text in pdf_texts:
    all_lines += text.split('\n')

clean_data = []

for line in all_lines:
    #match = re.findall(r'(Apr \d+)\s+(.+?)\s+(.+?)\s+\$(\d+\.\d+)', line)
    match = re.findall(r'(Apr \d+)\s+(\S+(?:\s+\S+)?)\s+(.+?)\s+\$(\d+\.\d+)', line)
    for m in match:
        clean_data.append(m)

df = pd.DataFrame(clean_data, columns=['Date','Merchant','Category','Amount'])
df['Amount'] = df['Amount'].astype(float)
df.head()

,Date,Merchant,Category,Amount
0,Apr 21,Best Buy,Shopping,142.29
1,Apr 13,Apple Store,Shopping,100.31
2,Apr 20,Apple Store,Shopping,101.50
3,Apr 20,Uber,Transport,291.30
4,Apr 14,Uber,Transport,129.09


## Save to Excel

In [ ]:
df.to_excel('cleaned_financial_data.xlsx', index=False)
print('Excel created successfully!')

Excel created successfully!


## Chunking

In [ ]:
chunks = df.astype(str).apply(lambda row: ' | '.join(row), axis=1).tolist()
chunks[:5]

['Apr 21 | Best Buy | Shopping | 142.29',
 'Apr 13 | Apple Store | Shopping | 100.31',
 'Apr 20 | Apple Store | Shopping | 101.5',
 'Apr 20 | Uber | Transport | 291.3',
 'Apr 14 | Uber | Transport | 129.09']

## Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks)
len(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

61

## Semantic Search

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query = 'food spending'
q_emb = model.encode([query])
similarities = cosine_similarity(q_emb, embeddings)

top_idx = similarities[0].argsort()[-5:]

for i in top_idx:
    print(chunks[i])

Apr 11 | Costco | Groceries | 79.63
Apr 17 | Whole Foods | Groceries | 66.59
Apr 10 | Chipotle | Food | 267.45
Apr 29 | Whole Foods | Groceries | 298.4
Apr 12 | Costco | Groceries | 39.57
